In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.append(r"E:\Projet_Alzheimer\configs")
from config_projet import C

prep = sorted(C.COHORTE.glob("preparation_modele_*"))[-1]
jeu = pd.read_csv(prep / "jeu_modelisation_359.csv")
print("Donnees :", prep.name, "|", jeu.shape)

# Variables connues a la visite de depart (aucune information du futur).
TAB_CANDIDATES = [
    "sexe", "education", "age_reference", "APOE_e4",
    "CDR_global", "CDR_SB", "MMSE_debut", "ADAS13", "FAQ_debut",
    "ABETA42", "TAU", "PTAU", "dx_debut",
]
FEATURES = [c for c in TAB_CANDIDATES if jeu[c].notna().sum() > 0]
vides = [c for c in TAB_CANDIDATES if c not in FEATURES]
if vides:
    print("Variables ignorees (vides) :", vides)

donnees = jeu.dropna(subset=["progression"]).copy()
X = donnees[FEATURES]
y = donnees["progression"].astype(int)
print("Participants :", len(y), "| progressent :", int(y.sum()),
      f"({y.mean() * 100:.1f}%)")
print("Variables utilisees :", FEATURES)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

imp = lambda: SimpleImputer(strategy="median")

def avec_echelle(clf):
    # Pour les modeles sensibles a l'echelle (logistique, SVM, kNN).
    return Pipeline([("imp", imp()), ("sc", StandardScaler()), ("clf", clf)])

def sans_echelle(clf):
    # Pour les modeles d'arbres (echelle sans importance).
    return Pipeline([("imp", imp()), ("clf", clf)])

MODELES = {
    "Hasard (reference)": sans_echelle(DummyClassifier(strategy="most_frequent")),
    "Regression logistique": avec_echelle(
        LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Arbre de decision": sans_echelle(
        DecisionTreeClassifier(class_weight="balanced", random_state=0)),
    "Random forest": sans_echelle(
        RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=0)),
    "Gradient boosting": sans_echelle(
        HistGradientBoostingClassifier(random_state=0)),
    "SVM (noyau RBF)": avec_echelle(
        SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=0)),
    "k plus proches voisins": avec_echelle(
        KNeighborsClassifier(n_neighbors=15)),
}

SCORES = ["roc_auc", "balanced_accuracy", "f1", "accuracy"]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [ ]:
# Evaluer chaque modele en validation croisee.
brut = {}
for nom, modele in MODELES.items():
    res = cross_validate(modele, X, y, cv=CV, scoring=SCORES)
    brut[nom] = {s: (res["test_" + s].mean(), res["test_" + s].std()) for s in SCORES}

# Tableau lisible : "moyenne ± ecart-type", trie par ROC-AUC.
noms_fr = {
    "roc_auc": "ROC-AUC", "balanced_accuracy": "Exactitude equilibree",
    "f1": "F1", "accuracy": "Exactitude",
}
tableau = pd.DataFrame({
    noms_fr[s]: {nom: f"{brut[nom][s][0]:.3f} ± {brut[nom][s][1]:.3f}" for nom in MODELES}
    for s in SCORES
})
ordre = sorted(MODELES, key=lambda n: brut[n]["roc_auc"][0], reverse=True)
tableau = tableau.loc[ordre]
tableau.index.name = "Modele"

print("=== Comparaison des modeles — progression (359, prevision statique) ===")
display(tableau)

In [ ]:
# Graphique : ROC-AUC de chaque modele avec barre d'erreur (pour la presentation).
noms = ordre
moyennes = [brut[n]["roc_auc"][0] for n in noms]
ecarts = [brut[n]["roc_auc"][1] for n in noms]

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
y_pos = np.arange(len(noms))[::-1]
ax.barh(y_pos, moyennes, xerr=ecarts, color="#4C72B0", capsize=4)
ax.set_yticks(y_pos); ax.set_yticklabels(noms)
ax.set_xlabel("ROC-AUC (validation croisee 5 plis)")
ax.set_xlim(0.45, 1.0)
ax.axvline(0.5, color="grey", linestyle="--", linewidth=1, label="hasard")
for yp, m in zip(y_pos, moyennes):
    ax.text(m + 0.01, yp, f"{m:.3f}", va="center", fontsize=9)
ax.set_title("Comparaison des modeles — prediction de la progression")
ax.legend(loc="lower right")

SORTIE = C.sortie("comparaison_modeles")
fig.savefig(SORTIE / "comparaison_AUC.png", dpi=150)
plt.show()
print("Figure enregistree :", SORTIE / "comparaison_AUC.png")

In [ ]:
# --- Choix final argumente ---
# Criteres, dans l'ordre : 1) ROC-AUC la plus elevee ;
# 2) stabilite (petit ecart-type) ; 3) simplicite / interpretabilite.
reels = {n: brut[n]["roc_auc"] for n in MODELES if n != "Hasard (reference)"}
meilleur = max(reels, key=lambda n: reels[n][0])
auc_m, std_m = reels[meilleur]

# La regression logistique est privilegiee si elle est tres proche du meilleur
# (<= 0,010 d'AUC) car elle est simple et interpretable.
auc_log = reels["Regression logistique"][0]
if meilleur != "Regression logistique" and (auc_m - auc_log) <= 0.010:
    choix, raison = ("Regression logistique",
        f"a seulement {auc_m - auc_log:.3f} d'AUC du meilleur ({meilleur}), "
        "mais plus simple et interpretable")
else:
    choix, raison = (meilleur, "meilleure ROC-AUC")

print("Classement par ROC-AUC :")
for n in ordre:
    if n == "Hasard (reference)":
        continue
    print(f"  {n:28s} AUC = {brut[n]['roc_auc'][0]:.3f} ± {brut[n]['roc_auc'][1]:.3f}")

print(f"\n>>> MODELE RETENU : {choix}")
print(f"    Justification : {raison}.")
print("    Criteres : 1) ROC-AUC  2) stabilite (ecart-type)  3) simplicite.")

# Enregistrer les resultats.
tableau.to_csv(SORTIE / "comparaison_modeles.csv", encoding="utf-8-sig")
pd.DataFrame(
    [{"modele": n, "roc_auc_moyenne": brut[n]["roc_auc"][0],
      "roc_auc_ecart_type": brut[n]["roc_auc"][1]} for n in ordre]
).to_csv(SORTIE / "classement_AUC.csv", index=False, encoding="utf-8-sig")
print("\nResultats enregistres dans :", SORTIE)